# La surchauffe en un chiffre : postes vacants par chômeur · *Overheating in one number: job openings per unemployed*

Notebook compagnon du chapitre **8. La carte du voyage : de la COVID au soft landing (2020-2025)** — [lire l'article](https://nmlab.io/ressources/de-la-covid-au-soft-landing-2020-2025).
Companion notebook to chapter **8. The Map of the Journey: From COVID to the Soft Landing (2020-2025)** — [read the article](https://nmlab.io/en/ressources/from-covid-to-the-soft-landing-2020-2025).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_ratio() -> Series:
    """Postes vacants (JOLTS, JTSJOL) rapportés au nombre de chômeurs (UNEMPLOY),
    2019-2025, en direct de FRED.
    JOLTS job openings over the number of unemployed, 2019-2025, live from FRED."""
    openings = nm.load_fred("JTSJOL", start="2019")
    unemployed = nm.load_fred("UNEMPLOY", start="2019")
    return (openings / unemployed).loc["2019":"2025"].dropna()

ratio = load_ratio()


from pandas import Timestamp as T
import matplotlib.dates as mdates
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="La surchauffe en un chiffre : postes vacants par chômeur",
        equil="équilibre : 1 pour 1",
        peak="mars 2022 : 2 postes vacants\npar chômeur",
        trough="avr. 2020 : 0,2",
        end="fin 2025 : 0,9",
        note="Sources : BLS via FRED — offres d'emploi JOLTS (JTSJOL) rapportées au nombre de chômeurs (UNEMPLOY). Mensuel, 2019-2025.\n"
             "Pic de mars 2022 : 2,0. La détente de 2022 à 2025 est passée par les offres, pas par les licenciements."),
    "en": dict(
        title="Overheating in one number: job openings per unemployed",
        equil="balance: 1 to 1",
        peak="Mar. 2022: 2 job openings\nper unemployed",
        trough="Apr. 2020: 0.2",
        end="end 2025: 0.9",
        note="Sources: BLS via FRED — JOLTS job openings (JTSJOL) over the number of unemployed (UNEMPLOY). Monthly, 2019-2025.\n"
             "March 2022 peak: 2.0. The 2022-2025 easing came through openings, not layoffs."),
}

def build_figure(ratio: Series, lang: str) -> Figure:
    """Ratio offres/chômeurs, ligne d'équilibre à 1, pic et creux annotés."""
    text = LABELS[lang]
    fig = nm.figure(height_px=1045)
    ax = nm.axes(fig, left=0.058, top=0.86)
    ax.axhline(1, color=nm.COLORS["muted"], linestyle=(0, (6, 4)), linewidth=1.8, alpha=0.85)
    ax.plot(ratio.index, ratio, color=nm.COLORS["blue"], linewidth=2.9)

    ax.set_ylim(0, 2.72)
    ax.set_yticks([0.0, 0.5, 1.0, 1.5, 2.0, 2.5])
    ax.set_xlim(T("2019-01-01"), T("2026-01-01"))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.text(T("2019-02-01"), 0.92, text["equil"], fontsize=20, color=nm.COLORS["muted"], va="top")
    peak = ratio.loc["2021":"2023"].idxmax()
    trough = ratio.loc["2020"].idxmin()
    ax.annotate(text["peak"], xy=(peak, ratio.loc[peak]), xytext=(T("2020-06-01"), 2.36),
                fontsize=21.5, color=nm.COLORS["text"], va="center", ha="left", linespacing=1.5,
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))
    ax.annotate(text["trough"], xy=(trough, ratio.loc[trough]), xytext=(T("2020-08-01"), 0.28),
                fontsize=21.5, color=nm.COLORS["text"], va="center", ha="left",
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))
    ax.annotate(text["end"], xy=(ratio.index[-1], ratio.iloc[-1]), xytext=(T("2024-06-01"), 1.42),
                fontsize=21.5, color=nm.COLORS["text"], va="center", ha="left",
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))
    nm.header(fig, text["title"])
    nm.footer(fig, text["note"])
    return fig

build_figure(ratio, LANG)